### Script for the imputation of missing values (bIRI tissues)
#### sqMSI data

For metabolites with <20% missing values

In [5]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

import warnings
warnings.filterwarnings("ignore")

nieromics_dir = "/exports/humgen/bmanzato/nieromics_dir"

In [6]:
# qMSI coord
qmsi_coord1 = pd.read_csv(f"{nieromics_dir}/qMSI_data/coord/20240720 mouse IRI kidney bIRI_1_xycoord.csv",index_col=0,comment="#",sep=";")
qmsi_coord2 = pd.read_csv(f"{nieromics_dir}/qMSI_data/coord/20240720 mouse IRI kidney bIRI_2_xycoord.csv",index_col=0,comment="#",sep=";")
qmsi_coord3 = pd.read_csv(f"{nieromics_dir}/qMSI_data/coord/20240720 mouse IRI kidney bIRI_3_xycoord.csv",index_col=0,comment="#",sep=";")

qmsi_coord = pd.concat([qmsi_coord1,qmsi_coord2,qmsi_coord3],axis=0)

# SD annotation
sd_ann = pd.read_csv(f"{nieromics_dir}/banksy_output/banksy_qmsi_output/bIRI_banksy_QMSI_RPCA_lambda1_kgeom8_res1_agftrue_scalefalse_130lipids_NOgaps.csv",index_col=0)
sd_ann.rename(columns={'clust_M1_lam1_k50_res1': 'SD'}, inplace=True)

# CT annotation
ct_ann = pd.read_csv(f"{nieromics_dir}/rosalie/qmsi_analysis/ct_annotations.csv",index_col=0)
ct_ann.rename(columns={'x': 'CT'}, inplace=True)
ct_ann_biri = ct_ann.iloc[:qmsi_coord.shape[0],:] # biri ct
qmsi_coord.index = ct_ann_biri.index
ct_ann_biri = ct_ann_biri[ct_ann_biri['CT'] != 'gaps'] # filter out gaps

meta = pd.concat([ct_ann_biri,sd_ann],axis=1)

# filter qmsi_coord to keep spots that are not gaps
qmsi_coord = qmsi_coord[qmsi_coord.index.isin(ct_ann_biri.index)]
qmsi_coord.head()

,x,y
Spot 127359,2985.473877,-52.851521
Spot 127360,3005.473877,-52.851521
Spot 127361,3025.473877,-52.851521
Spot 127362,3045.473877,-52.851521
Spot 127363,3065.473877,-52.851521


In [7]:
# Count Matrix (newly normalized qMSI values)
qmsi_cm = pd.read_csv(f"{nieromics_dir}/rosalie/qmsi_analysis/normalized_values_biri-1712.csv",index_col=0)

# filter (biri only)
qmsi_cm = qmsi_cm[qmsi_cm['spot'].isin(meta.index)]

# pivot the df
pivoted_qmsi_cm = qmsi_cm.pivot(index='spot', columns='annotation', values='normalized_value')
pivoted_qmsi_cm.head()

annotation,(2Z)-2-aminobut-2-enoate/2-iminobutanoate,"(6S,7R)-2,6,7,8-Tetrahydroxy-2-oxo-1,3,2lambda5-dioxaphosphonan-5-one/Inositol cyclic phosphate",(E)-but-2-enedioate;hydron/Maleic acid,"2,3-Dihydroxybutanoic acid/(2-Hydroxyethoxy)acetic acid/D-Erythrose/L-Erythrulose/Erythrose/A,b-Dihydroxyisobutyric acid/4-Deoxythreonic acid/4-Deoxyerythronic acid/2,4-Dihydroxybutanoic acid/(S)-3,4-Dihydroxybutyric acid",2-Furanmethanol,2-Hexyldecanoic acid/FFA(16:0),2-Phosphonopropionic acid,Acetylphosphate/Phosphonoacetate,Adenosine monophosphate/Deoxyguanosine 5'-monophosphate(dGMP)/3'-Adenylic acid,Alanine/Sarcosine,...,PS 36:2,Pyruvaldehyde,Stearic acid,Succinic acid,Succinic acid semialdehyde/2-Ketobutyric acid/Acetoacetic acid,Succinic anhydride,Uridine-5'-monophosphate,gamma-amino-gamma-cyanobutanoic acid,glycolic acid,phosphonoacetaldehyde/phosphonoacetaldehyde
spot,,,,,,,,,,,,,,,,,,,,,
Spot 127359,NaN,371.546287,182.355629,1155.718495,2414.019665,590.248257,2152.167430,254.139934,513.367397,101.048457,...,212.703706,1867.763264,643.625568,2306.747199,562.572947,1567.203997,274.882288,871.589202,2365.558944,2898.515138
Spot 127360,125.881153,720.104129,247.485839,982.492702,2608.419421,923.158428,3811.035605,467.000779,631.311400,111.680509,...,405.773224,1889.938819,1494.780309,1371.917301,874.861719,2112.348173,NaN,552.817027,4097.638372,4020.903447
Spot 127361,340.327586,1058.273511,180.295114,2041.802545,2594.734023,1049.352619,5743.335501,693.858431,802.714842,172.375491,...,416.146003,2004.598450,1562.048091,2320.061300,812.394113,2908.610390,262.760879,1895.300928,5415.263150,6089.652249
Spot 127362,381.801449,1368.404358,255.317111,1244.380423,4482.171308,1486.963104,5990.226764,740.785581,1151.973120,153.283951,...,420.732606,2290.615799,1255.092505,2811.536779,859.500299,1518.095636,388.019168,1018.490058,3759.481744,6200.648287
Spot 127363,NaN,1431.173056,175.364707,NaN,6269.887044,1195.155484,3929.642289,745.092080,1091.829294,254.808169,...,236.445757,2532.039512,1518.843828,4105.687747,1125.061366,2629.395480,NaN,1537.544770,2626.210191,4564.004077


In [8]:
def drop_features_with_high_nan(pivoted_qmsi_cm, threshold=0.2):
    nan_percentage = pivoted_qmsi_cm.isna().mean()
    filtered_df = pivoted_qmsi_cm.loc[:, nan_percentage <= threshold]
    return filtered_df

pivoted_qmsi_cm = drop_features_with_high_nan(pivoted_qmsi_cm, threshold=0.2)

In [9]:
pivoted_qmsi_cm.shape

(91980, 63)

#### Imputation process

For each missing value (NaN), the function searches for valid neighboring values:

First, it checks if any of the 8 immediate neighbors have valid values (non-NaN).

If no valid values are found among the 8 neighbors, it then checks the 24 neighbors.

If valid neighbors are found, the function imputes the missing value using the mean of the available neighbors.

If neither the 8 nor 24 neighbors provide valid values, the function imputes the missing value using the overall mean of the entire metabolite (column).

In [10]:
def impute_missing_values(coords, data):

    step_size = 20
    
    # 8 neighbor offsets
    offsets_8 = [(dx * step_size, dy * step_size) for dx in [-1, 1, 0] for dy in [-1, 1, 0] if not (dx == 0 and dy == 0)]
    
    # 24 neighbor offsets (neighbors up to distance 2)
    offsets_24 = [(dx * step_size, dy * step_size) for dx in range(-2, 3) for dy in range(-2, 3) if not (dx == 0 and dy == 0)]
    
    # convert the coordinates df to a numpy array for fast indexing
    coords_array = coords[['x', 'y']].values
    
    # convert the df to numpy for faster access
    data_values = data.values
    
    coord_to_index = {tuple(coord): idx for idx, coord in enumerate(coords_array)}
    
    # initialize
    count_numbers = 0
    count_all_nbh_NaN = 0
    count_mean_of_8 = 0
    count_mean_of_24 = 0
    count_mean_of_all = 0
    
    # create a df to track if a value is imputed or not 
    status_df = pd.DataFrame("original", index=data.index, columns=data.columns)
    
    for col_idx in range(data_values.shape[1]):

        missing_idx = np.isnan(data_values[:, col_idx])
        
        # for each missing value find its neighbors
        for idx in np.where(missing_idx)[0]:
            x, y = coords_array[idx]
            
            neighbors_8 = []
            for dx, dy in offsets_8:
                neighbor_coord = (x + dx, y + dy)
                if neighbor_coord in coord_to_index:
                    neighbor_idx = coord_to_index[neighbor_coord]
                    neighbor_value = data_values[neighbor_idx, col_idx]
                    if not np.isnan(neighbor_value):
                        neighbors_8.append(neighbor_value)
            
            neighbors_24 = []
            if not neighbors_8:  # check 24 neighbors if no 8 neighbors are available
                for dx, dy in offsets_24:
                    neighbor_coord = (x + dx, y + dy)
                    if neighbor_coord in coord_to_index:
                        neighbor_idx = coord_to_index[neighbor_coord]
                        neighbor_value = data_values[neighbor_idx, col_idx]
                        if not np.isnan(neighbor_value):
                            neighbors_24.append(neighbor_value)
            
            if neighbors_8:
                data_values[idx, col_idx] = np.mean(neighbors_8)
                status_df.iloc[idx, col_idx] = "imputed"  
                count_mean_of_8 += 1
            elif neighbors_24:
                data_values[idx, col_idx] = np.mean(neighbors_24)
                status_df.iloc[idx, col_idx] = "imputed"  
                count_mean_of_24 += 1
            else:
                # impute with overall mean if all neighbors are nan
                overall_mean = np.nanmean(data_values[:, col_idx]) # mean
                data_values[idx, col_idx] = overall_mean
                status_df.iloc[idx, col_idx] = "imputed"  
                count_mean_of_all += 1

    imputed_data = pd.DataFrame(data_values, columns=data.columns, index=data.index)
    
    # statistics
    print(f"NaN transformed to numbers using the mean of 8 neighbors: {count_mean_of_8}")
    print(f"NaN transformed to numbers using the mean of 24 neighbors: {count_mean_of_24}")
    print(f"NaN transformed to the overall mean of the metabolite: {count_mean_of_all}")
    
    return imputed_data, status_df

In [11]:
imputed_df, status_df = impute_missing_values(qmsi_coord, pivoted_qmsi_cm)

NaN transformed to numbers using the mean of 8 neighbors: 284411
NaN transformed to numbers using the mean of 24 neighbors: 1074
NaN transformed to the overall mean of the metabolite: 429


In [15]:
imputed_df.to_csv(f"{nieromics_dir}/qMSI_data/imputed_qMSI_data/qMSI_countmatrix_imputed.csv")

In [14]:
status_df.to_csv(f"{nieromics_dir}/qMSI_data/imputed_qMSI_data/status_df_imputed.csv")